# 05 · Directional resampling power
Protocol §9: power on the **binary run scale** for each of the 9 preregistered contrasts. Each simulated study draws the number
of eligible confirmatory scenarios (counted on the frozen reference-agent runs of the confirmatory split: fault-free stratum for
H1/H2a/b, drift-exposed scenarios for H2c) from the pilot's eligible scenarios with their agent rate in the reference package,
moves each rate toward the boundary so that the mean shift equals a preregistered smallest effect of interest (5 or 10 points)
**in the expected direction**, draws 3 repetitions x 2 modes (or x packages) of binary runs, and tests at the strictest Holm
threshold (alpha = .05/9); only rejections with the expected sign count (H3b: either sign). NaN = not estimable from the pilot
(too few scenarios, or a pilot rate that leaves no room for the effect).
Decision rule: if power at 10 points is < .80 for any H1 contrast or for H3a, add scenarios before confirmatory runs.

**Sensitivity (v1.3a).** The simulation spreads the effect evenly over scenarios. The pilot's stage effects sat in a few
scenarios, so `power_*_pilotsd` recomputes power with a noncentral t using the pilot's own SD of the item differences. If the
two disagree, the decision is the authors' (protocol §9): add scenarios, or keep 120 and treat null H1/H2 verdicts as
inconclusive.

In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "harness"), str(ROOT / "benchmark")]
import pandas as pd
from IPython.display import Markdown, Image, display
pd.set_option("display.max_colwidth", 120)
print("repository root:", ROOT)

repository root: C:\IEEE-RESEARCH-WORK


In [2]:
import analyze
df = analyze.load(ROOT / "pilot/runs_dev_qwen3-8b.jsonl")
ideal = analyze.load(ROOT / "pilot/runs_dev_ideal.jsonl")
conf = analyze.load(ROOT / "pilot/runs_confirmatory_ideal.jsonl")
power = analyze.power(df, ideal, conf)
power.to_csv(ROOT / "pilot/analysis/power.csv", index=False)
display(power)
rule = power[((power.hyp == "H1") | ((power.hyp == "H3") & (power.outcome == "ABCR"))) & ~(power.power_10pt >= 0.8)]
print("Decision rule (protocol §9): H1/H3a contrasts below .80 power at 10 points:",
      rule[["hyp", "outcome", "contrast", "power_10pt"]].values.tolist() or "none")
sens = power[((power.hyp == "H1") | ((power.hyp == "H3") & (power.outcome == "ABCR"))) & ~(power.power_10pt_pilotsd >= 0.8)]
print("Same rule with the pilot's item SD (sensitivity):",
      sens[["hyp", "outcome", "contrast", "power_10pt_pilotsd"]].round(2).values.tolist() or "none")

Decision rule (protocol §9): H1/H3a contrasts below .80 power at 10 points: none
Same rule with the pilot's item SD (sensitivity): [['H1', 'SAR', 'integration - production', 0.53], ['H1', 'SuAR', 'development - production', 0.3], ['H1', 'SuAR', 'integration - production', 0.2]]


,hyp,outcome,contrast,method,expected,pilot_items,pilot_effect,pilot_rate_a,confirm_items,runs_per_cell,power_5pt,power_10pt,pilot_sd,power_5pt_pilotsd,power_10pt_pilotsd
0,H1,SAR,development - production,faultfree,-1,14,-0.054,1.000,66,6,0.99725,1.00000,0.200,0.210137,0.876850
1,H1,SAR,integration - production,faultfree,-1,14,-0.054,0.982,68,6,0.82500,0.99775,0.280,0.089413,0.533212
2,H1,SuAR,development - production,faultfree,-1,14,0.036,0.857,66,6,0.99425,1.00000,0.352,0.049332,0.297933
3,H1,SuAR,integration - production,faultfree,-1,14,0.054,0.875,68,6,0.61950,0.98600,0.418,0.034790,0.195798
4,H2,Accuracy,postrelease - production,faultfree,-1,18,-0.042,0.861,84,6,0.89900,1.00000,0.324,0.082306,0.497197
5,H2,SAR,postrelease - production,faultfree,-1,18,-0.181,1.000,84,6,0.99975,1.00000,0.341,0.072208,0.441720
6,H2,ABCR,postrelease - production,drift,-1,20,-0.350,0.963,41,6,0.45500,0.91350,0.469,0.016923,0.070597
7,H3,ABCR,tool - prompt,raw,1,28,0.060,0.798,115,9,0.71525,1.00000,0.122,0.939830,1.000000
8,H3,Accuracy,tool - prompt,raw,0,24,0.023,0.667,96,15,0.69600,0.99950,0.086,0.997364,1.000000
